# 03 · Gold · `dim_uc`
### V-Energy Grid 360 — Arquitetura Medalhão

Grava `v_energy.gold.dim_uc`: a "fonte única de verdade" sobre cada cliente (seção 3 do case). **Grão: 1 linha = 1 UC (45.000 linhas).**

**Responde a** (contrato da Gold, A1): P4 (quem é o cliente, medidor, painel solar), P10 (filtros e o exemplo "UCs com painel solar em Olinda com queda acima de 30%"), P12 (filtros do portal), P1 (GD, principal causa da queda de volume) e P6 (perfil do medidor e da GD para a fila de perdas).

| Bloco de colunas | Fonte (Silver) |
|---|---|
| Cadastro, endereço, classe, alimentador, ligação | `unidades_consumidoras` |
| Medidor ativo, idade, troca recente | `medidores` |
| Painel solar (GD) | `geracao_distribuida` |
| Primeiro e último mês, consumo médio dos 12 meses | `leituras` |

**Decisões deste notebook:**
- **LGPD:** nome e documento saem **só mascarados**. As colunas originais não entram na Gold.
- **Chave:** `id_uc`, sem chave substituta (`uc_sk`). Desvio consciente do padrão da aula: o `id_uc` é único, estável e é a chave usada em todas as Silvers e Gold. Uma chave numérica extra só obrigaria todas as fatos a fazer mais um join.
- **Datas de referência:** idade do medidor e janelas de 12 meses calculadas até o fim da base, **2026-06-30**.

- **Log de qualidade:** as checagens vão para `silver.dq_log`, o mesmo log das Silvers e da `fato_consumo_faturamento`, com `table_name = gold.dim_uc`.

**Pré-requisito:** as 4 Silvers gravadas.

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from pyspark.sql import functions as F, Row
from datetime import datetime

tabela_gold = f"{gold_schema}.dim_uc"
tabela_fato = f"{gold_schema}.fato_consumo_faturamento"   # só para o teste de conexão (seção 12)
tabela_dq_log = f"{silver_schema}.dq_log"                 # mesmo log das Silvers e da fato_consumo_faturamento
FIM_BASE = "2026-06-30"
P12_INI, P12_FIM = "2025-07", "2026-06"          # últimos 12 meses
P12_ANT_INI, P12_ANT_FIM = "2024-07", "2025-06"  # 12 meses anteriores
TROCA_INI, TROCA_FIM = "2025-09-01", "2026-03-31"  # concentração de trocas de medidor

def n_br(n):
    return f"{n:,}".replace(",", ".")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
print(f"destino: {tabela_gold}")

## 2. Funções de checagem de qualidade

Mesmas funções das Silvers. Os resultados vão para `silver.dq_log`, o mesmo log das Silvers e da `fato_consumo_faturamento`, com as mesmas colunas, e a tabela só é gravada se **todas** as checagens passarem.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total,
                          failed_rows=failed, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total,
                          failed_rows=dupes, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def dq_check_contagem(table_name: str, check_name: str, df, esperado: int):
    """Checagem de contagem total: passa só se o número de linhas for exatamente o esperado."""
    total = df.count()
    diferenca = abs(total - esperado)
    passed = diferenca == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total,
                          failed_rows=diferenca, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {n_br(total)} linhas (esperado {n_br(esperado)})")

## 3. Leitura das Silvers

In [0]:
ucs = spark.table(f"{silver_schema}.unidades_consumidoras")
med = spark.table(f"{silver_schema}.medidores")
gd = spark.table(f"{silver_schema}.geracao_distribuida")
leit = spark.table(f"{silver_schema}.leituras").select("id_uc", "mes_referencia", "consumo_kwh")

for nome, df in [("unidades_consumidoras", ucs), ("medidores", med), ("geracao_distribuida", gd)]:
    print(f"{nome}: {n_br(df.count())} linhas")  # esperado: 45.000 | 51.299 | 3.510

## 4. Cadastro com dados pessoais mascarados (LGPD)

**Regra:** CPF/CNPJ e nome do titular **nunca** saem da Silver sem máscara (contrato da Gold, convenções).
- **Nome:** primeiro nome + iniciais dos demais (`EDVALDO PEREIRA BARBOSA` → `EDVALDO P*** B***`). Permite ao atendente confirmar o cliente ao telefone sem expor o nome completo.
- **Documento:** só os dígitos do meio. CPF `***.816.240-**`; CNPJ `**.***.816/0001-**`.

As colunas originais (`nome_titular`, `cpf_cnpj`) **não entram** na Gold.

In [0]:
nome_mascarado = F.expr("""
    concat_ws(' ',
        element_at(split(nome_titular, ' '), 1),
        concat_ws(' ', transform(slice(split(nome_titular, ' '), 2, greatest(size(split(nome_titular, ' ')) - 1, 0)),
                                 x -> concat(substr(x, 1, 1), '***'))))
""")
doc_mascarado = (F.when(F.col("tipo_pessoa") == "PF",
                        F.concat(F.lit("***."), F.substring("cpf_cnpj", 4, 3), F.lit("."), F.substring("cpf_cnpj", 7, 3), F.lit("-**")))
                  .when(F.col("tipo_pessoa") == "PJ",
                        F.concat(F.lit("**.***."), F.substring("cpf_cnpj", 6, 3), F.lit("/"), F.substring("cpf_cnpj", 9, 4), F.lit("-**"))))

cadastro = ucs.select(
    "id_uc",
    nome_mascarado.alias("nome_titular_mascarado"),
    doc_mascarado.alias("documento_mascarado"),
    "tipo_pessoa", "logradouro", "numero", "bairro", "municipio", "uf", "cep",
    F.col("latitude").cast("decimal(9,6)").alias("latitude"),
    F.col("longitude").cast("decimal(9,6)").alias("longitude"),
    "classe", "subgrupo_tarifario", "alimentador", "data_ligacao", "data_ligacao_estimada", "status",
)
display(cadastro.select("id_uc", "nome_titular_mascarado", "documento_mascarado", "tipo_pessoa").limit(5))

## 5. Medidor ativo e troca recente

- **Medidor ativo:** cada UC tem exatamente 1 medidor `ATIVO` (garantido na Silver de UCs).
- **Idade:** anos entre a instalação e o fim da base.
- **Troca recente:** a UC teve um medidor `RETIRADO` entre set/2025 e mar/2026, período em que se concentram as 6.299 trocas (fator explicativo da visão 360 e do diagnóstico).

In [0]:
ativo = (med.filter("status = 'ATIVO'")
         .select("id_uc",
                 F.col("id_medidor").alias("id_medidor_ativo"),
                 F.col("tipo").alias("tipo_medidor_ativo"),
                 F.col("modelo").alias("modelo_medidor_ativo"),
                 F.col("data_instalacao").alias("data_instalacao_medidor"))
         .withColumn("idade_medidor_anos",
                     F.round(F.datediff(F.lit(FIM_BASE).cast("date"), "data_instalacao_medidor") / 365.25, 1).cast("decimal(5,1)")))

trocou = (med.filter(F.col("status") == "RETIRADO")
             .filter(F.col("data_retirada").between(F.lit(TROCA_INI).cast("date"), F.lit(TROCA_FIM).cast("date")))
             .select("id_uc").distinct()
             .withColumn("medidor_trocado_recentemente", F.lit(True)))

print("UCs com medidor ativo:", n_br(ativo.count()))           # esperado: 45.000
print("UCs com troca recente:", n_br(trocou.count()))           # esperado: perto de 6.299

## 6. Geração distribuída (painel solar)

Vem direto da Silver de GD (1 linha por UC). UCs sem GD ficam com as colunas de GD nulas e as flags `false`.
`energia_injetada_media_kwh` **não deve ser usada como medida** quando `energia_injetada_atipica = true` (log G5).

In [0]:
gd_cols = gd.select(
    "id_uc",
    F.lit(True).alias("possui_gd"),
    F.col("potencia_kwp").alias("potencia_gd_kwp"),
    F.col("data_conexao").alias("data_conexao_gd"),
    F.col("modalidade").alias("modalidade_gd"),
    F.col("energia_injetada_kwh_media_mensal").alias("energia_injetada_media_kwh"),
    "energia_injetada_atipica",
)
print("UCs com GD:", n_br(gd_cols.count()))  # esperado: 3.510

## 7. Leituras: período e consumo dos últimos 12 meses

- `primeiro_mes_leitura` e `ultimo_mes_leitura`: período coberto pela medição da UC.
- `consumo_medio_12m_kwh`: média mensal de `2025-07` a `2026-06`.
- `consumo_medio_12m_anterior_kwh`: média mensal de `2024-07` a `2025-06`.
- `variacao_consumo_12m_pct`: variação entre os dois. Nula se a UC não tem leitura no período anterior (ex.: ligada depois de jun/2025).

É o que responde ao exemplo da Patrícia ("UCs com painel solar em Olinda com queda acima de 30%") e alimenta a fila de perdas.

In [0]:
periodo = leit.groupBy("id_uc").agg(F.min("mes_referencia").alias("primeiro_mes_leitura"),
                                    F.max("mes_referencia").alias("ultimo_mes_leitura"))

consumo = (leit.filter(F.col("mes_referencia").between(P12_ANT_INI, P12_FIM))
               .groupBy("id_uc").agg(
                   F.avg(F.when(F.col("mes_referencia").between(P12_INI, P12_FIM), F.col("consumo_kwh"))).alias("c12"),
                   F.avg(F.when(F.col("mes_referencia").between(P12_ANT_INI, P12_ANT_FIM), F.col("consumo_kwh"))).alias("c12a"))
               .select("id_uc",
                       F.round("c12", 2).cast("decimal(12,2)").alias("consumo_medio_12m_kwh"),
                       F.round("c12a", 2).cast("decimal(12,2)").alias("consumo_medio_12m_anterior_kwh"),
                       F.when(F.col("c12a") > 0, F.round((F.col("c12") / F.col("c12a") - 1) * 100, 2))
                        .cast("decimal(8,2)").alias("variacao_consumo_12m_pct")))
print("UCs com leitura:", n_br(periodo.count()))  # esperado: 45.000

## 8. Montagem da `dim_uc`

Todas as junções partem do cadastro (45.000 UCs) com `left join`, para nenhuma UC se perder. A ordem das colunas segue o contrato da Gold.

In [0]:
dim_uc = (
    cadastro
    .join(ativo, "id_uc", "left")
    .join(trocou, "id_uc", "left")
    .join(gd_cols, "id_uc", "left")
    .join(periodo, "id_uc", "left")
    .join(consumo, "id_uc", "left")
    .withColumn("medidor_trocado_recentemente", F.coalesce("medidor_trocado_recentemente", F.lit(False)))
    .withColumn("possui_gd", F.coalesce("possui_gd", F.lit(False)))
    .withColumn("energia_injetada_atipica", F.coalesce("energia_injetada_atipica", F.lit(False)))
    .select(
        "id_uc", "nome_titular_mascarado", "documento_mascarado", "tipo_pessoa",
        "logradouro", "numero", "bairro", "municipio", "uf", "cep", "latitude", "longitude",
        "classe", "subgrupo_tarifario", "alimentador", "data_ligacao", "data_ligacao_estimada", "status",
        "id_medidor_ativo", "tipo_medidor_ativo", "modelo_medidor_ativo", "data_instalacao_medidor",
        "idade_medidor_anos", "medidor_trocado_recentemente",
        "possui_gd", "potencia_gd_kwp", "data_conexao_gd", "modalidade_gd",
        "energia_injetada_media_kwh", "energia_injetada_atipica",
        "primeiro_mes_leitura", "ultimo_mes_leitura",
        "consumo_medio_12m_kwh", "consumo_medio_12m_anterior_kwh", "variacao_consumo_12m_pct",
    )
)
dim_uc.printSchema()

## 9. Checagens de qualidade

| Checagem | Protege |
|---|---|
| 45.000 linhas e `id_uc` único | nenhuma UC perdida ou duplicada nas junções |
| Nenhum dado pessoal sem máscara | LGPD |
| Toda UC com medidor ativo e período de leitura | junções corretas |
| 3.510 UCs com GD; GD com potência e data | junção com a Silver de GD |
| Flags nunca nulas | filtros do portal e do agente |
| Idade do medidor e consumo não negativos | cálculos derivados |
| 14 municípios, 297 pares município + bairro, 122 alimentadores | números do case; nenhum valor de endereço ou rede perdido ou inventado na montagem |

In [0]:
t = "gold.dim_uc"
dq_results.clear()

def ok(cond):
    return F.coalesce(cond, F.lit(False))

dq_check_contagem(t, "total de linhas = 45.000", dim_uc, 45_000)
dq_check_unique(t, "id_uc único", dim_uc, ["id_uc"])

# LGPD: nenhuma coluna com nome ou documento completos
dq_check(t, "colunas originais de dados pessoais ausentes", dim_uc.limit(1),
         F.lit(not any(c in dim_uc.columns for c in ["nome_titular", "cpf_cnpj"])))
dq_check(t, "documento mascarado no formato CPF ou CNPJ", dim_uc,
         ok(F.col("documento_mascarado").rlike(r"^(\*\*\*\.[0-9]{3}\.[0-9]{3}-\*\*|\*\*\.\*\*\*\.[0-9]{3}/[0-9]{4}-\*\*)$")))
dq_check(t, "nome mascarado (sobrenomes só com a inicial)", dim_uc,
         ok(~F.col("nome_titular_mascarado").rlike(r" [A-Z]{2,}")))

for coluna in ["bairro", "municipio", "classe", "alimentador", "data_ligacao", "id_medidor_ativo",
               "tipo_medidor_ativo", "idade_medidor_anos", "primeiro_mes_leitura", "consumo_medio_12m_kwh",
               "medidor_trocado_recentemente", "possui_gd", "energia_injetada_atipica"]:
    dq_check(t, f"{coluna} não nulo", dim_uc, F.col(coluna).isNotNull())

dq_check_contagem(t, "UCs com GD = 3.510", dim_uc.filter("possui_gd"), 3_510)
dq_check(t, "UC com GD tem potência e data de conexão", dim_uc.filter("possui_gd"),
         F.col("potencia_gd_kwp").isNotNull() & F.col("data_conexao_gd").isNotNull())
dq_check(t, "UC sem GD não tem potência", dim_uc.filter("NOT possui_gd"), F.col("potencia_gd_kwp").isNull())
dq_check(t, "idade do medidor não negativa", dim_uc, ok(F.col("idade_medidor_anos") >= 0))
dq_check(t, "consumo médio não negativo", dim_uc, ok(F.col("consumo_medio_12m_kwh") >= 0))

# Números do case (bairro só vale junto com o município)
dq_check_contagem(t, "municípios distintos = 14", dim_uc.select("municipio").distinct(), 14)
dq_check_contagem(t, "pares município + bairro = 297", dim_uc.select("municipio", "bairro").distinct(), 297)
dq_check_contagem(t, "alimentadores distintos = 122", dim_uc.select("alimentador").distinct(), 122)

## 10. Registro das checagens em `silver.dq_log`

Gravado **sempre**, mesmo com falha (modo `append`).

In [0]:
spark.createDataFrame(dq_results).write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(tabela_dq_log)
display(spark.table(tabela_dq_log).filter(F.col("table_name") == t)
        .orderBy(F.col("checked_at").desc()).limit(len(dq_results)))

## 11. Gravação da Gold, só se todas as checagens passarem

In [0]:
falhas = [r for r in dq_results if not r.passed]
if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_gold} NÃO foi gravada:")
    for r in falhas:
        print(f"  - {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"{tabela_gold} não gravada: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

dim_uc.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_gold)
print(f"Todas as {len(dq_results)} checagens passaram. {tabela_gold} gravada: {n_br(spark.table(tabela_gold).count())} linhas")

## 12. Conferências e consultas de teste

Mostram que a tabela responde às perguntas do contrato e conferem números do case. O exemplo da Patrícia é uma pergunta real que o agente de IA vai receber.

O primeiro bloco testa a **conexão com a `gold.fato_consumo_faturamento`** (Marcela). Só roda se ela existir no mesmo workspace; se não existir, avisa e segue. Esperado: 0 em todas as linhas de "fora" e "diferente", e 3.510 UCs com GD nas duas tabelas.

In [0]:
g = spark.table(tabela_gold)

# Conexão com a gold.fato_consumo_faturamento (só se ela existir neste workspace)
if spark.catalog.tableExists(tabela_fato):
    fato = spark.table(tabela_fato)
    ucs_fato = fato.select("id_uc").distinct()
    print("UCs da fato fora da dim_uc:", n_br(ucs_fato.join(g, "id_uc", "left_anti").count()))           # esperado: 0
    print("UCs da dim_uc sem linha na fato:", n_br(g.join(ucs_fato, "id_uc", "left_anti").count()))      # esperado: 0
    gd_fato = fato.filter("possui_gd_no_mes").select("id_uc").distinct().count()
    print("UCs com GD -> dim_uc:", n_br(g.filter("possui_gd").count()), "| fato (algum mês com GD):", n_br(gd_fato))  # esperado: 3.510 e 3.510
    attrs = (fato.select("id_uc", "classe", "municipio", "bairro").distinct().alias("f")
             .join(g.select("id_uc", "classe", "municipio", "bairro").alias("d"), "id_uc"))
    print("UCs com classe, município ou bairro diferente entre fato e dim_uc:",
          n_br(attrs.filter(~(F.col("f.classe").eqNullSafe(F.col("d.classe"))
                              & F.col("f.municipio").eqNullSafe(F.col("d.municipio"))
                              & F.col("f.bairro").eqNullSafe(F.col("d.bairro"))))
               .select("id_uc").distinct().count()))                                                     # esperado: 0
else:
    print(f"{tabela_fato} não existe neste workspace: teste de conexão fica para quem tiver as duas tabelas")

# Números do case: 45.000 UCs, 14 municípios, "quase 300 bairros", "mais de 120 alimentadores"
display(g.agg(F.count("*").alias("ucs"),
              F.countDistinct("municipio").alias("municipios"),
              F.countDistinct("municipio", "bairro").alias("bairros_municipio_bairro"),
              F.countDistinct("alimentador").alias("alimentadores")))

# P10 (Patrícia): quantas UCs com painel solar em Olinda tiveram queda de consumo acima de 30%?
print("UCs com GD em Olinda e queda acima de 30%:",
      g.filter("possui_gd AND municipio = 'Olinda' AND variacao_consumo_12m_pct < -30").count())

# P4 (Camila): visão 360 de uma UC com GD e troca recente de medidor
display(g.filter("possui_gd AND medidor_trocado_recentemente").limit(3))

# Perfil da base: classe x GD
display(g.groupBy("classe").agg(F.count("*").alias("ucs"), F.sum(F.col("possui_gd").cast("int")).alias("com_gd"),
                                F.round(F.avg("consumo_medio_12m_kwh"), 1).alias("consumo_medio_12m")).orderBy("classe"))